# 矩阵分解与重构

学习目标：用 QR、SVD 和对称矩阵特征分解表示小矩阵，通过形状、重构和正交性检查结果，并辨认各方法的输入条件。

前置知识：矩阵乘法、共轭转置、秩、特征值、正交性、数值容差。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

全部矩阵均为本章构造的小输入，后续单元沿用首次导入的 np。浮点检查统一使用 rtol=0、atol=1e-12，作为这些小型、适度量级示例的验收约定；不把该阈值推广到任意矩阵。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 QR 分解与重构

QR 分解把矩阵表示为 Q @ R：Q 的列两两正交且长度为 1，R 是上三角或相应的上梯形矩阵。下面把三行两列的数组分解，再将结果相乘，核对是否还原输入。

本例使用默认的缩减模式 reduced，Q 的形状为 (3, 2)，R 为 (2, 2)。norm(..., ord="fro") 计算矩阵 Frobenius 范数，即各元素绝对值平方和的平方根，这里用于汇总重构差异。

In [1]:
import numpy as np

matrix = np.array([[1.0, 1.0], [1.0, 0.0], [0.0, 1.0]])
q, r = np.linalg.qr(matrix, mode="reduced")
restored = q @ r

print(q.shape, r.shape, restored.shape)  # (3, 2) (2, 2) (3, 2)。
print(restored)  # 在舍入误差内还原三行 [1 1]、[1 0]、[0 1]。
print(np.linalg.norm(restored - matrix, ord="fro"))  # 接近 0 的实际重构误差。
print(np.allclose(restored, matrix, rtol=0, atol=1e-12))  # True。

(3, 2) (2, 2) (3, 2)
[[ 1.00000000e+00  1.00000000e+00]
 [ 1.00000000e+00 -1.78835871e-16]
 [ 0.00000000e+00  1.00000000e+00]]
6.716134489646879e-16
True


## 2 QR 的形状与正交性

### 2.1 缩减模式

设输入矩阵形状为 (m, n)，m 为行数、n 为列数，k=min(m, n)。reduced 模式返回形状 (m, k) 的 Q 和 (k, n) 的 R。

实数 Q 的正交列满足 Q.T @ Q 接近 k 阶单位矩阵。Q 不一定是方阵，因此不能把 Q @ Q.T 也一律当成单位矩阵。下面继续检查上一单元的 Q 与 R。

In [2]:
print(np.allclose(q.T @ q, np.eye(2), rtol=0, atol=1e-12))  # True，列正交归一。
print(np.allclose(q @ q.T, np.eye(3), rtol=0, atol=1e-12))  # False，缩减 Q 不是方阵。
print(r)  # (2, 2) 上三角矩阵，左下元素为 0。
print(np.allclose(r[1, 0], 0, rtol=0, atol=1e-12))  # True。

True
False
[[-1.41421356 -0.70710678]
 [ 0.          1.22474487]]
True


### 2.2 完整模式

complete 模式返回形状 (m, m) 的 Q 和 (m, n) 的 R。对于行多于列的输入，它保留更多正交列；两种模式都能重构原矩阵。

检查分解应关注重构与正交性质，不把某一组因子的符号当作唯一答案。下面重新给出输入，并明确两种模式的尺寸。

In [3]:
matrix = np.array([[1.0, 1.0], [1.0, 0.0], [0.0, 1.0]])
q_full, r_full = np.linalg.qr(matrix, mode="complete")

print(q_full.shape, r_full.shape)  # (3, 3) (3, 2)。
print(np.allclose(q_full @ r_full, matrix, rtol=0, atol=1e-12))  # True。
print(np.allclose(q_full.T @ q_full, np.eye(3), rtol=0, atol=1e-12))  # True。

(3, 3) (3, 2)
True
True


## 3 奇异值分解

### 3.1 缩减 SVD

奇异值分解（singular value decomposition，SVD）把二维矩阵表示为 U @ diag(s) @ Vh。s 是按降序排列的非负奇异值；Vh 已经是右奇异向量矩阵的共轭转置，重构时不能再把它转置一次。

对于形状 (m, n) 的输入，令 k=min(m, n)。full_matrices=False 返回 U 的形状 (m, k)、s 的形状 (k,)、Vh 的形状 (k, n)。下面用可手算的矩形对角布局核对奇异值与重构。

In [4]:
matrix = np.array([[3.0, 0.0], [0.0, 2.0], [0.0, 0.0]])
u, singular_values, vh = np.linalg.svd(matrix, full_matrices=False)
restored = (u * singular_values) @ vh

print(u.shape, singular_values.shape, vh.shape)  # (3, 2) (2,) (2, 2)。
print(singular_values)  # [3. 2.]，非负且降序。
print(restored)  # 还原输入；u * singular_values 按列缩放，等价于 u @ diag(s)。
print(np.allclose(restored, matrix, rtol=0, atol=1e-12))  # True。
print(np.allclose(u.T @ u, np.eye(2), rtol=0, atol=1e-12))  # True。
print(np.allclose(vh @ vh.T, np.eye(2), rtol=0, atol=1e-12))  # True。

(3, 2) (2,) (2, 2)
[3. 2.]
[[3. 0.]
 [0. 2.]
 [0. 0.]]
True
True
True


### 3.2 完整 SVD

默认 full_matrices=True 返回形状 (m, m) 的 U 和 (n, n) 的 Vh，而 s 仍只有 k 个奇异值。矩形输入下，直接拿完整 U 乘 k×k 对角阵可能尺寸不匹配。

下面沿用上一单元的 matrix。构造形状 (m, n) 的矩形 Sigma，把奇异值放在主对角线上，就能按完整尺寸重构。

In [5]:
u_full, singular_values, vh_full = np.linalg.svd(matrix, full_matrices=True)
sigma = np.zeros(matrix.shape, dtype=np.float64)
sigma[:2, :2] = np.diag(singular_values)
restored = u_full @ sigma @ vh_full

print(u_full.shape, sigma.shape, vh_full.shape)  # (3, 3) (3, 2) (2, 2)。
print(np.allclose(restored, matrix, rtol=0, atol=1e-12))  # True。
print(np.linalg.norm(restored - matrix, ord="fro"))  # 本例重构误差为 0。
print(np.linalg.svd(matrix, compute_uv=False))  # [3. 2.]，只需奇异值时可不计算向量。

(3, 3) (3, 2) (2, 2)
True
0.0
[3. 2.]


## 4 数值秩与阈值

matrix_rank() 按大于阈值的奇异值数量判断数值秩。默认阈值结合最大奇异值、矩阵尺寸和该类型的机器精度；它用于容忍数值计算误差，不等于测量任务的误差模型。

tol 指定绝对阈值。下面两个奇异值为 1 和 1e-10：是否将第二个方向视为有效，取决于阈值。改变阈值会改变数值秩，但没有改变原数组。

In [6]:
matrix = np.diag(np.array([1.0, 1e-10]))
singular_values = np.linalg.svd(matrix, compute_uv=False)
default_tol = singular_values.max() * max(matrix.shape) * np.finfo(matrix.dtype).eps

print(singular_values)  # [1.e+00 1.e-10]。
print(default_tol)  # 约 4.44e-16，本例默认阈值小于第二个奇异值。
print(np.linalg.matrix_rank(matrix))  # 2。
print(np.linalg.matrix_rank(matrix, tol=1e-8))  # 1，小方向低于指定阈值。
print(np.linalg.matrix_rank(matrix, tol=1e-12))  # 2。
print(np.count_nonzero(singular_values > 1e-8))  # 1，与相同阈值的判定一致。

[1.e+00 1.e-10]
4.440892098500626e-16
2
1
2
1


## 5 对称矩阵的特征分解

### 5.1 eigh 的特征值与特征向量

eigh() 用于实对称矩阵或复数 Hermitian 矩阵，返回升序特征值和按列排列的特征向量。Hermitian 表示矩阵等于自身的共轭转置。

对特征值 w 及其向量 v，有 A @ v = w * v，其中 A 是输入方阵。下面二维实对称矩阵的两个特征值为 1、3。将各特征向量组成 V，可用 (V * w) @ V.T 重构实对称矩阵。

In [7]:
matrix = np.array([[2.0, 1.0], [1.0, 2.0]])
eigenvalues, eigenvectors = np.linalg.eigh(matrix)
restored = (eigenvectors * eigenvalues) @ eigenvectors.T

print(eigenvalues)  # [1. 3.]，升序排列。
print(eigenvectors.shape)  # (2, 2)，每一列对应一个特征值。
print(np.allclose(matrix @ eigenvectors, eigenvectors * eigenvalues, rtol=0, atol=1e-12))
# True，所有列都满足特征方程。
print(np.allclose(eigenvectors.T @ eigenvectors, np.eye(2), rtol=0, atol=1e-12))  # True。
print(np.allclose(restored, matrix, rtol=0, atol=1e-12))  # True。

[1. 3.]
(2, 2)
True
True
True


### 5.2 共轭转置

复数矩阵的重构与正交检查需要共轭转置，写作 conjugate().T。普通 .T 只交换轴，不把虚部变号。

下面矩阵的两个非对角元素互为共轭，特征值仍为实数。向量的具体相位不是这里的验收目标，应检查它满足的方程。

In [8]:
matrix = np.array([[2.0, 1.0j], [-1.0j, 2.0]], dtype=np.complex128)
eigenvalues, vectors = np.linalg.eigh(matrix)
adjoint = vectors.conjugate().T

print(np.allclose(matrix, matrix.conjugate().T, rtol=0, atol=1e-12))  # True，Hermitian。
print(eigenvalues)  # [1. 3.]，特征值为实数。
print(np.allclose(adjoint @ vectors, np.eye(2), rtol=0, atol=1e-12))  # True。
print(np.allclose((vectors * eigenvalues) @ adjoint, matrix, rtol=0, atol=1e-12))  # True。
print(np.allclose((vectors * eigenvalues) @ vectors.T, matrix, rtol=0, atol=1e-12))
# False，普通转置不能替代复数共轭转置。

True
[1. 3.]
True
True
False


### 5.3 输入条件需要另行检查

eigh() 通过 UPLO 选择使用下三角或上三角，默认使用下三角；复数对角线的虚部会被视为零。调用成功不能证明输入原本就是 Hermitian。

下面故意提供不对称输入，比较两种三角选择的重构。实际任务应在调用前按数据约定检查对称性，而不是依赖函数替你检查。

In [9]:
matrix = np.array([[2.0, 9.0], [1.0, 2.0]])
print(np.allclose(matrix, matrix.T, rtol=0, atol=1e-12))  # False，输入不对称。

lower_values, lower_vectors = np.linalg.eigh(matrix, UPLO="L")
upper_values, upper_vectors = np.linalg.eigh(matrix, UPLO="U")
print((lower_vectors * lower_values) @ lower_vectors.T)
# 重构 [[2, 1], [1, 2]]，使用了下三角的信息。
print((upper_vectors * upper_values) @ upper_vectors.T)
# 重构 [[2, 9], [9, 2]]，使用了上三角的信息。
print(np.allclose((lower_vectors * lower_values) @ lower_vectors.T, matrix, rtol=0, atol=1e-12))
# False，分解成功没有还原原来的非对称输入。

False
[[2. 1.]
 [1. 2.]]
[[2. 9.]
 [9. 2.]]
False


## 6 选学：Cholesky 分解

Cholesky 要求输入为 Hermitian 正定矩阵；实数情形即对称正定。默认返回下三角 L，使 A = L @ L.conjugate().T。正定条件比仅仅对称更强，不满足时可能抛出 LinAlgError。

函数也不会完整检查 Hermitian 条件，只使用所选三角部分。下面先核对对称性，再对一个已知正定的小矩阵重构，并观察含零特征值的对角矩阵失败。

In [10]:
matrix = np.array([[4.0, 2.0], [2.0, 3.0]])
assert np.allclose(matrix, matrix.T, rtol=0, atol=1e-12)
lower = np.linalg.cholesky(matrix)

print(lower)  # 两行约为 [2, 0]、[1, 1.41421356]。
print(np.allclose(lower @ lower.T, matrix, rtol=0, atol=1e-12))  # True。

# 预期 LinAlgError：对角矩阵含零特征值，半正定不足以满足 Cholesky 的正定条件。
np.linalg.cholesky(np.diag([1.0, 0.0]))

[[2.         0.        ]
 [1.         1.41421356]]
True


LinAlgError: Matrix is not positive definite

## 7 选学：一般方阵的 eig

eig() 处理一般方阵，特征值不保证排序，也可能有非零虚部。NumPy 2.5 起，eig 的返回数组统一使用复数类型；特征向量按列排列，但不保证彼此正交，甚至可能不构成满秩矩阵。因此不能直接照搬 eigh 的正交重构公式。

下面实数旋转矩阵的特征值为一对纯虚数。核对特征方程，不依赖返回顺序或向量相位。

In [11]:
matrix = np.array([[0.0, -1.0], [1.0, 0.0]])
eigenvalues, eigenvectors = np.linalg.eig(matrix)

print(eigenvalues)  # 两个值为 +1j 与 -1j，不把顺序作为通用保证。
print(eigenvectors.shape, eigenvectors.dtype)  # (2, 2) complex128。
print(np.allclose(matrix @ eigenvectors, eigenvectors * eigenvalues, rtol=0, atol=1e-12))
# True，实数输入也可能需要复数特征向量。

[0.+1.j 0.-1.j]
(2, 2) complex128
True


## 本章小结

（1）QR 与 SVD 都能重构矩阵，输出形状取决于输入尺寸和模式；缩减因子不一定是方阵。

（2）SVD 返回 Vh，已经包含共轭转置；数值秩取决于奇异值阈值。

（3）eigh 适用于实对称或 Hermitian 矩阵；复数重构与正交检查要使用共轭转置。

（4）验收要检查输入条件、因子形状、重构误差与必要的正交性，不以调用成功或因子符号完全一致作为判据。

## 练习

（1）对下面三行两列矩阵进行缩减 QR 分解，预测 Q 与 R 的形状，检查重构与 Q 的列正交性。再改为完整模式，说明增加了哪些维度。

In [12]:
matrix = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])

# 在此先记录形状预测，再分解；检查重构、正交性，使用本章约定容差。
# 不把某一个符号选择或某个因子的逐元素值作为唯一正确答案。

（2）只需要三行两列矩阵的两个奇异值，以及能够重构输入的最小尺寸因子。选择 SVD 参数并解释理由；如果改为只报告数值秩，还需要说明什么阈值条件？

In [13]:
matrix = np.array([[4.0, 0.0], [0.0, 1e-9], [0.0, 0.0]])

# 在此选择 full_matrices 并重构；奇异值应为 [4, 1e-9]。
# 比较 tol=1e-8 与 tol=1e-10 下的秩，说明改变阈值为何会改变结论。

（3）下面输入是复数 Hermitian 矩阵。求特征值，分别用普通转置和共轭转置重构，说明应接受哪一种结果及其理由。

In [14]:
matrix = np.array([[3.0, 2.0j], [-2.0j, 3.0]], dtype=np.complex128)

# 在此先核对 Hermitian 条件，再使用 eigh；特征值应接近 [1, 5]。
# 检查特征方程、向量正交性和重构误差，不只看输出是否为实数。

（4）选学：两个对称矩阵都能使用 Cholesky 吗？先按给出的对角元素判断条件，再分别运行核对，并解释失败输入的异常原因。

In [15]:
positive = np.diag([2.0, 3.0])
nonpositive = np.diag([2.0, -1.0])

# 在此说明正定条件，再分别调用；成功结果检查 L @ L.T。
# 将两个输入分别放在独立单元运行，解释失败时的 LinAlgError。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（NumPy 2.5） | [qr](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.qr.html) 的分解定义、reduced／complete 形状与正交列；[svd](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.svd.html) 的 U、s、Vh、缩减／完整形状、重构与 compute_uv；[matrix_rank](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.matrix_rank.html) 的 tol、默认阈值及 Notes 中测量误差与数值秩；[eigh](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.eigh.html) 的 Hermitian 条件、升序特征值、列向量、UPLO 与对角虚部；[cholesky](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.cholesky.html) 的正定条件、三角选择、不完整检查与 LinAlgError；[eig](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.eig.html) 的顺序、特征方程及 Notes 中向量不保证独立／正交；[2.5 发布说明](https://numpy.org/doc/2.5/release/2.5.0-notes.html) 的 linalg.eig and linalg.eigvals now always return complex arrays（API 页面旧返回类型描述以此更新为准）；[norm](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.norm.html) 的 Frobenius 范数；[allclose](https://numpy.org/doc/2.5/reference/generated/numpy.allclose.html) 的容差比较规则。 |